# Zero-Shot Forest Point Cloud Segmentation

This notebook demonstrates the **forest-panoptic-nav** zero-shot segmentation pipeline.
No training data or ML models are needed. The pipeline uses purely geometric features
(RANSAC ground estimation, DBSCAN clustering, and shape analysis) to classify forest
point clouds into FinnWoodlands-compatible panoptic labels.

**Semantic classes:**

| ID | Class    | Type   | Description                        |
|----|----------|--------|------------------------------------|
| 0  | Unlabeled| Stuff  | Unclassified points                |
| 1  | Ground   | Stuff  | Terrain surface                    |
| 2  | Track    | Stuff  | Forest path / trail                |
| 4  | Spruce   | Thing  | Spruce tree trunk (conical crown)  |
| 5  | Birch    | Thing  | Birch tree trunk (smooth bark)     |
| 6  | Pine     | Thing  | Pine tree trunk (tall, thick)      |
| 7  | Obstacle | Thing  | Rocks, fallen logs, etc.           |

**Pipeline steps:**
1. RANSAC ground plane estimation on lowest points
2. Ground removal and track detection (flat, elongated regions)
3. Height filtering for trunk detection (0.5 m -- 8.0 m above ground)
4. DBSCAN clustering of above-ground points
5. Per-cluster geometric feature extraction (trunk radius, bark roughness, crown shape)
6. Species classification via combined geometric scoring
7. Traversability cost map generation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from forest_panoptic_nav.zero_shot import ZeroShotForestSegmenter
from forest_panoptic_nav.traversability import TraversabilityMapper

## 1. Create a Synthetic Forest Scene

We generate a synthetic point cloud with:
- A 20 m x 20 m ground plane
- A forest path (track) running through the scene
- 8 trees of varying height and radius placed around the path
- 1 obstacle (a boulder / fallen log)

In [ ]:
rng = np.random.default_rng(42)

# --- Ground plane: 20m x 20m with slight noise ---
n_ground = 5000
ground_xy = rng.uniform(0, 20, size=(n_ground, 2))
ground_z = rng.normal(0.0, 0.02, size=(n_ground, 1))  # nearly flat
ground_pts = np.hstack([ground_xy, ground_z])

# --- Track / path: a 2m wide strip along Y-axis at X~10 ---
n_track = 1500
track_x = rng.normal(10.0, 0.4, size=(n_track, 1))  # narrow in X
track_y = rng.uniform(0, 20, size=(n_track, 1))       # full length in Y
track_z = rng.normal(-0.02, 0.01, size=(n_track, 1))  # slightly depressed, very flat
track_pts = np.hstack([track_x, track_y, track_z])

# --- Trees: 8 cylinders of varying species ---
tree_configs = [
    # (x, y, radius, height, n_points) - mimicking different species
    (3.0,  4.0,  0.08, 4.0, 200),   # thin, short -> birch-like
    (5.0, 12.0,  0.07, 3.5, 180),   # thin, short -> birch-like
    (7.0,  8.0,  0.15, 5.5, 300),   # medium, medium -> spruce-like
    (14.0, 3.0,  0.18, 6.0, 350),   # medium-thick, tall -> spruce-like
    (16.0, 9.0,  0.22, 7.0, 400),   # thick, tall -> pine-like
    (13.0,16.0,  0.20, 6.5, 380),   # thick, tall -> pine-like
    (4.0, 17.0,  0.09, 3.0, 160),   # thin, short -> birch-like
    (17.0,17.0,  0.14, 5.0, 280),   # medium -> spruce-like
]

tree_points = []
for tx, ty, tr, th, tn in tree_configs:
    # Generate cylindrical trunk points
    angles = rng.uniform(0, 2 * np.pi, size=tn)
    radii = rng.normal(tr, tr * 0.1, size=tn)  # slight noise around radius
    z_vals = rng.uniform(0.5, th, size=tn)      # from 0.5m to tree height
    xs = tx + radii * np.cos(angles)
    ys = ty + radii * np.sin(angles)
    tree_points.append(np.column_stack([xs, ys, z_vals]))

tree_pts = np.vstack(tree_points)

# --- Obstacle: a low, wide cluster (boulder) ---
n_obstacle = 120
obs_center = np.array([12.0, 7.0])
obs_xy = rng.normal(obs_center, 0.6, size=(n_obstacle, 2))
obs_z = rng.uniform(0.5, 1.2, size=(n_obstacle, 1))  # low, wide
obstacle_pts = np.hstack([obs_xy, obs_z])

# --- Combine all points ---
points = np.vstack([ground_pts, track_pts, tree_pts, obstacle_pts]).astype(np.float32)
print(f"Total points: {len(points):,}")
print(f"  Ground:   {len(ground_pts):,}")
print(f"  Track:    {len(track_pts):,}")
print(f"  Trees:    {len(tree_pts):,}")
print(f"  Obstacle: {len(obstacle_pts):,}")

## 2. Run Zero-Shot Segmentation

The `ZeroShotForestSegmenter` processes the raw XYZ point cloud and produces
panoptic labels (semantic class + instance ID) without any trained model.

In [ ]:
segmenter = ZeroShotForestSegmenter()
result = segmenter.predict(points)

# Per-class point counts
class_names = {
    0: "Unlabeled",
    1: "Ground",
    2: "Track",
    3: "Lake",
    4: "Spruce",
    5: "Birch",
    6: "Pine",
    7: "Obstacle",
}

print("Semantic segmentation results:")
print("-" * 35)
for cls_id in sorted(np.unique(result.semantic_labels)):
    count = (result.semantic_labels == cls_id).sum()
    name = class_names.get(cls_id, f"Class {cls_id}")
    print(f"  {name:>12s} (id={cls_id}): {count:>6,} points")
print("-" * 35)
print(f"  {'Total':>12s}       : {len(result.semantic_labels):>6,} points")

## 3. Instance Detection Results

The segmenter also assigns instance IDs to individual "thing" objects
(tree trunks and obstacles). Each detected object gets a unique ID.

In [ ]:
print(f"Number of detected instances: {result.num_instances}")
print(f"Number of semantic classes present: {result.num_semantic_classes}")
print()

# Per-species instance breakdown
for species_id, species_name in [(4, "Spruce"), (5, "Birch"), (6, "Pine"), (7, "Obstacle")]:
    instances = result.get_instances(semantic_class=species_id)
    if instances:
        sizes = [len(inst) for inst in instances]
        print(f"  {species_name}: {len(instances)} instance(s), "
              f"points per instance: {sizes}")
    else:
        print(f"  {species_name}: 0 instances")

## 4. Generate Traversability Map

The `TraversabilityMapper` projects the segmented point cloud onto a 2D cost grid.
Each cell receives a cost based on the highest-cost semantic class present:

| Class     | Cost | Traversable |
|-----------|------|-------------|
| Track     | 0.0  | Yes         |
| Ground    | 0.1  | Yes         |
| Unlabeled | 0.5  | Maybe       |
| Trees     | 1.0  | No          |
| Obstacle  | 1.0  | No          |

In [ ]:
mapper = TraversabilityMapper(resolution=0.5, kernel_size=3)
cost_map = mapper.compute_cost_map(result.points, result.semantic_labels)

print(f"Cost map shape: {cost_map.grid.shape}")
print(f"Resolution: {cost_map.resolution} m/cell")
print(f"Origin (world): ({cost_map.origin[0]:.1f}, {cost_map.origin[1]:.1f}) m")
print(f"Traversable ratio: {cost_map.traversable_ratio:.1%}")

## 5. Visualization

Three views of the segmentation and traversability results:
1. **Top-down view** -- XY plane colored by semantic class
2. **Side view** -- XZ plane showing the height profile
3. **Traversability cost map** -- 2D heatmap of navigation cost

In [ ]:
# Color map for semantic classes
class_colors = {
    0: (0.7, 0.7, 0.7),   # gray      - Unlabeled
    1: (0.6, 0.4, 0.2),   # brown     - Ground
    2: (0.9, 0.8, 0.5),   # sandy     - Track
    3: (0.2, 0.4, 0.8),   # blue      - Lake
    4: (0.0, 0.5, 0.0),   # dark green- Spruce
    5: (0.6, 0.9, 0.3),   # lime      - Birch
    6: (0.0, 0.3, 0.0),   # forest    - Pine
    7: (0.8, 0.2, 0.2),   # red       - Obstacle
}

colors = np.array([class_colors.get(l, (0.5, 0.5, 0.5)) for l in result.semantic_labels])

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Top-down view (XY) ---
ax = axes[0]
ax.scatter(points[:, 0], points[:, 1], c=colors, s=1, alpha=0.6)
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_title("Top-Down View (colored by class)")
ax.set_aspect("equal")

# Legend
present = sorted(np.unique(result.semantic_labels))
handles = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=class_colors.get(c, (0.5, 0.5, 0.5)),
               markersize=8, label=class_names.get(c, f"Class {c}"))
    for c in present
]
ax.legend(handles=handles, loc="upper right", fontsize=8)

# --- Side view (XZ) ---
ax = axes[1]
ax.scatter(points[:, 0], points[:, 2], c=colors, s=1, alpha=0.6)
ax.set_xlabel("X (m)")
ax.set_ylabel("Z (m)")
ax.set_title("Side View (height profile)")

# --- Traversability cost map heatmap ---
ax = axes[2]
im = ax.imshow(
    cost_map.grid.T, origin="lower", cmap="RdYlGn_r",
    vmin=0, vmax=1, aspect="equal",
    extent=[
        cost_map.origin[0],
        cost_map.origin[0] + cost_map.grid.shape[0] * cost_map.resolution,
        cost_map.origin[1],
        cost_map.origin[1] + cost_map.grid.shape[1] * cost_map.resolution,
    ],
)
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_title("Traversability Cost Map")
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Cost (0=free, 1=blocked)")

plt.tight_layout()
plt.show()

## Summary

This demo showed the complete zero-shot forest panoptic segmentation pipeline:

| Step | Component                    | Description                                         |
|------|------------------------------|-----------------------------------------------------|
| 1    | RANSAC ground estimation     | Fits a plane to the lowest points                   |
| 2    | Track detection              | Identifies flat, depressed regions as forest paths   |
| 3    | DBSCAN clustering            | Groups above-ground points into individual objects   |
| 4    | Geometric feature extraction | Trunk radius, bark roughness, crown shape per cluster|
| 5    | Species classification       | Scores each cluster for Spruce / Birch / Pine        |
| 6    | Obstacle detection           | Low aspect-ratio clusters flagged as obstacles       |
| 7    | Traversability mapping       | Projects labels onto a 2D cost grid for planning     |

**Semantic class reference:**

| ID | Class     | Category | Key geometric features                     |
|----|-----------|----------|--------------------------------------------|
| 1  | Ground    | Stuff    | Below ground plane threshold               |
| 2  | Track     | Stuff    | Flat, slightly depressed ground regions     |
| 4  | Spruce    | Thing    | Medium trunk, rough bark, conical crown     |
| 5  | Birch     | Thing    | Thin trunk, smooth bark, rounded crown      |
| 6  | Pine      | Thing    | Thick trunk, tall, flat-topped crown        |
| 7  | Obstacle  | Thing    | Low aspect ratio, wide spread               |

No training data, labels, or GPU required -- the entire pipeline runs on CPU
using NumPy and scikit-learn.